In [1]:
import os 
import pandas as pd
import tensorflow as tf
import sionna
import pickle

# Set random seed for reproducibility
tf.random.set_seed(42)

# Commented out IPython magic to ensure Python compatibility.
import matplotlib.pyplot as plt
import numpy as np
import time
import json
import csv

# Import Sionna RT components
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, PathSolver, RadioMapSolver, subcarrier_frequencies

# For link-level simulations
from sionna.phy.channel import cir_to_ofdm_channel, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.phy.utils import ebnodb2no, PlotBER, dbm_to_watt, lin_to_db, log2, db_to_lin
from sionna.phy.ofdm import KBestDetector, LinearDetector, ResourceGrid
from sionna.phy.constants import BOLTZMANN_CONSTANT
from sionna.phy.mimo import StreamManagement

import copy

In [17]:
print('test')

test


In [2]:
scene = load_scene("toy_XML/Chalmers_entrance.xml")

In [3]:
for i, obj in enumerate(scene.objects.values()):
    print(f"{obj.name}: {obj.radio_material.name}")
    if i >= 10:
        break

Teknologg_rden: itu_medium_dry_ground
no-name-1: itu_concrete
no-name-2: itu_metal
Plane: itu_medium_dry_ground
no-name-3: itu_concrete
no-name-4: itu_metal


In [3]:
tx1 = Transmitter("tx1", [0.5, -3, 15], [0.0, 0.0, 0.0])   #Toymodel coordinates
scene.add(tx1)

#tx2 = Transmitter("tx2", [0.5, -343, 30], [0.0, 0.0, 0.0])   #Chalmers coordinates
#scene.add(tx2)

#tx3 = Transmitter("tx3", [65, -181.5, 35], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx3)

#tx4 = Transmitter("tx4", [22, 182, 28], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx4)

#tx5 = Transmitter("tx5", [-147, 327, 28], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx5)

#tx6 = Transmitter("tx6", [-212, -33, 15], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(tx6)

rx = Receiver("rx", [16, -10, 1.5], [0.0, 0.0, 0.0])
scene.add(rx)   #toy example coordinates

rx2 = Receiver("rx2", [-46, 28.5, 1.4], [0.0, 0.0, 0.0])
scene.add(rx2)  #toy example coordinates

rx3 = Receiver("rx3", [-74, 67, 1.6], [0.0, 0.0, 0.0])
scene.add(rx3)  #toy example coordinates

#rx4 = Receiver("rx4", [1.3, 9, 1.4], [0.0, 0.0, 0.0])
#scene.add(rx4)  #toy example coordinates

#rx5 = Receiver("rx5", [70, -29, 1.6], [0.0, 0.0, 0.0])
#scene.add(rx5)  #toy example coordinates

#rx6 = Receiver("rx6", [-7.6, 28.5, 1.4], [0.0, 0.0, 0.0])
#scene.add(rx6)  #toy example coordinates

##rx8 = Receiver("rx8", [60, -294, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx8)

#rx9 = Receiver("rx9", [89, -169, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx9)

#rx10 = Receiver("rx10", [55.6, -26, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx10)

#rx11 = Receiver("rx11", [-35, 106, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx11)

#rx12 = Receiver("rx12", [-70, 0.3, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx12)

#rx13 = Receiver("rx13", [-116, 229, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx13)

#rx14 = Receiver("rx14", [-253, 402, 1.5], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx14)

#rx15 = Receiver("rx15", [-235, 25, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx15)

#rx16 = Receiver("rx16", [-248, -52, 1.6], [0.0, 0.0, 0.0])  #Chalmers coordinates
#scene.add(rx16)


In [6]:
# Set the carrier frequency 
fc = 3.5e9 # from band n78 in 3GPP TS 38.104. Can be changed later when testing out mmWave or sub-THz
scene.frequency = fc
# Define the wavelength of the signal and calculate the antenna distance 
c = 3e8
Lambda = c/fc
antenna_dist = Lambda/2

scene.tx_array = PlanarArray(num_rows=4, num_cols=4, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="tr38901", polarization="cross") # changed it to cross-polarization as described in 3GPP TR 38.901
# Configure antenna array for all receivers
scene.rx_array = PlanarArray(num_rows=2, num_cols=2, vertical_spacing=antenna_dist, horizontal_spacing=antenna_dist, pattern="tr38901", polarization="cross")

p_solver = PathSolver() 
paths = p_solver(scene=scene, max_depth=4, los=True, specular_reflection=True, diffuse_reflection=True, refraction=False)



In [7]:
scene.preview(paths=paths)

Renderer(camera=PerspectiveCamera(aspect=1.31, children=(DirectionalLight(intensity=0.25, matrixWorldNeedsUpda…

In [8]:
paths.interactions

[[[[1, 0, 0]],
  [[0, 0, 0]],
  [[0, 1, 1]]],
 [[[1, 0, 0]],
  [[0, 0, 0]],
  [[0, 0, 0]]],
 [[[0, 0, 0]],
  [[0, 0, 0]],
  [[0, 0, 0]]],
 [[[0, 0, 0]],
  [[0, 0, 0]],
  [[0, 0, 0]]]]

In [ ]:
#scene.render("preview", coverage_map=cm);

In [ ]:
#scene.preview(paths=paths)

In [26]:
image = scene.render("preview", paths=paths);
#image.savefig("image_10.png")

TypeError: Scene.render() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given